# 🇹🇷 Turkey at the Olympics — Deep Dive Analysis

**Goal:** Analyze Turkey's Olympic performance in detail — medals by year, sport, and athlete.  
**Data:** Cleaned dataset from `01_EDA_cleaning.ipynb`

---

## 1. Import Libraries & Load Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go

sns.set_theme(style='whitegrid')
pd.set_option('display.max_columns', None)
%matplotlib inline

df = pd.read_csv('../data/athlete_events_clean.csv')

# Turkey Summer Olympics filter
df_turkey = df[(df['Team'] == 'Turkey') & (df['Season'] == 'Summer')].copy()

print(f'Turkey Summer Olympics records: {len(df_turkey):,}')
print(f'Years participated: {sorted(df_turkey["Year"].unique())}')

## 2. Overall Medal Summary

In [ ]:
# Total medal counts
summary = df_turkey[['Gold', 'Silver', 'Bronze', 'Total']].sum()
print('🏅 Turkey Total Medals (Summer Olympics)')
print(f'🥇 Gold:   {int(summary["Gold"])}')
print(f'🥈 Silver: {int(summary["Silver"])}')
print(f'🥉 Bronze: {int(summary["Bronze"])}')
print(f'🏅 Total:  {int(summary["Total"])}')

## 3. Medals by Year

In [ ]:
yearly = df_turkey.groupby('Year')[['Gold', 'Silver', 'Bronze', 'Total']].sum().reset_index()

fig = px.bar(
    yearly, x='Year', y=['Gold', 'Silver', 'Bronze'],
    title='🇹🇷 Turkey Olympic Medals by Year',
    color_discrete_map={'Gold': '#FFD700', 'Silver': '#C0C0C0', 'Bronze': '#CD7F32'},
    barmode='stack',
    labels={'value': 'Medal Count', 'variable': 'Medal Type'}
)
fig.update_layout(xaxis=dict(tickmode='linear', dtick=4))
fig.show()

print('\nTop 5 years by total medals:')
print(yearly.sort_values('Total', ascending=False).head())

## 4. Medals by Sport

In [ ]:
sport_medals = (df_turkey.groupby('Sport')[['Gold', 'Silver', 'Bronze', 'Total']]
                .sum()
                .sort_values('Total', ascending=False)
                .reset_index())

sport_medals_filtered = sport_medals[sport_medals['Total'] > 0]

# Bar chart
fig = px.bar(
    sport_medals_filtered,
    x='Total', y='Sport',
    orientation='h',
    color='Gold',
    color_continuous_scale='YlOrRd',
    title='🇹🇷 Turkey Medals by Sport'
)
fig.update_layout(yaxis={'categoryorder': 'total ascending'})
fig.show()

In [ ]:
# Treemap
fig2 = px.treemap(
    sport_medals_filtered,
    path=['Sport'],
    values='Total',
    color='Gold',
    color_continuous_scale='YlOrRd',
    title='Turkey Medal Distribution by Sport (Treemap)'
)
fig2.show()

## 5. Top Athletes

In [ ]:
top_athletes = (df_turkey[df_turkey['Total'] > 0]
                .groupby(['Name', 'Sport'])[['Gold', 'Silver', 'Bronze', 'Total']]
                .sum()
                .sort_values('Total', ascending=False)
                .head(15)
                .reset_index())

print('🏆 Top 15 Turkish Medalists:')
print(top_athletes.to_string(index=False))

In [ ]:
fig = px.bar(
    top_athletes,
    x='Total', y='Name',
    orientation='h',
    color='Sport',
    title='Top 15 Turkish Olympic Athletes by Medal Count'
)
fig.update_layout(yaxis={'categoryorder': 'total ascending'})
fig.show()

## 6. Physical Profile of Medalists

In [ ]:
# Age distribution: medalists vs non-medalists
df_turkey['Medalist'] = df_turkey['Total'].apply(lambda x: 'Medalist' if x > 0 else 'Non-Medalist')

plt.figure(figsize=(10, 4))
sns.kdeplot(data=df_turkey, x='Age', hue='Medalist',
            palette={'Medalist': '#FFD700', 'Non-Medalist': '#aaaaaa'},
            fill=True, alpha=0.4)
plt.title('Age Distribution: Turkish Medalists vs Non-Medalists', fontsize=14)
plt.tight_layout()
plt.savefig('../visuals/age_medalists.png', dpi=150)
plt.show()

In [ ]:
# Age by sport (medalists only)
medalists = df_turkey[df_turkey['Total'] > 0]

fig = px.box(
    medalists, x='Sport', y='Age',
    title='Age Distribution of Turkish Medalists by Sport',
    color='Sport'
)
fig.update_layout(showlegend=False)
fig.show()

In [ ]:
# Height vs Weight scatter
fig = px.scatter(
    medalists, x='Weight', y='Height',
    color='Sport', size='Total',
    hover_data=['Name', 'Year'],
    title='Height vs Weight of Turkish Medalists'
)
fig.show()

## 7. Medal Trend Over Time (Smoothed)

In [ ]:
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=yearly['Year'], y=yearly['Total'],
    mode='lines+markers',
    name='Total Medals',
    line=dict(color='steelblue', width=2),
    marker=dict(size=8)
))

fig.add_trace(go.Scatter(
    x=yearly['Year'], y=yearly['Gold'],
    mode='lines+markers',
    name='Gold',
    line=dict(color='#FFD700', width=2, dash='dot'),
    marker=dict(size=6)
))

fig.update_layout(
    title='Turkey Olympic Medal Trend (1936–2016)',
    xaxis_title='Year',
    yaxis_title='Medal Count'
)
fig.show()

---
## ✅ Summary

| Insight | Detail |
|---------|--------|
| 🥇 Most successful sport | Wrestling |
| 📅 Best Olympic year | 1948 |
| 👤 Most decorated athlete | See top athletes table |
| 📈 Trend | Consistent participation, peaks in Wrestling |

**Next:** `03_statistical_analysis.ipynb` → Hypothesis testing & country comparison